In [1]:
from langchain.chat_models import ChatOpenAI
from dotenv import load_dotenv, find_dotenv
import os

from sqlalchemy.engine import URL
from langchain.agents import create_sql_agent
from langchain.utilities import SQLDatabase
from langchain_experimental.sql import SQLDatabaseChain
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.agents.agent_types import AgentType
from langchain.llms import OpenAI

In [2]:
# Load the environment variables
load_dotenv(find_dotenv())

True

In [3]:
# Create llm object
llm = OpenAI(temperature=0)

In [4]:
# create database uri
connectionURL = URL.create(
    drivername="postgresql+psycopg2",
    host=os.environ.get("HOST"),
    database="enqurious_etl_db",
    username=os.environ.get("USER"),
    port=5432,
    password=os.environ.get("PASSWORD"),
)

In [5]:
# Create sqldatabase object from URI
db = SQLDatabase.from_uri(connectionURL)

In [8]:
# print(f"All the available methods of SQL Database object: {db.__dir__()}")
# print(f"Database contains the attribute _all_tables that gives all the available table in database\
#       as shown: {db._all_tables}")

# print(db.table_info)

**The agent should contain the SQLDatabaseToolkit which contains tools to:**

1. Create and execute queries
2. Check query syntax
3. Retrieve table descriptions
4. ... and more

In [6]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

In [10]:
# # Use create_sql_agent
# agent_executor = create_sql_agent(
#     llm=llm,
#     toolkit=toolkit,
#     verbose=True,
#     handle_parsing_errors=True,
#     return_intermediate_steps=True,
# )

In [7]:
agent_executor = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    handle_parsing_errors=True,
    return_intermediate_steps=True,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
)

In [16]:
agent_executor.run("How many tables are present in the database?")



> Entering new AgentExecutor chain...
Action: sql_db_list_tables
Action Input: 
Observation: clients, feedback_fact, learners_fact, order_duration_fact, orders_dimension, progress_fact, progress_report, skills_fact, table_meta_data
Thought: I now know the number of tables in the database
Final Answer: There are 9 tables in the database.

> Finished chain.


'There are 9 tables in the database.'

In [17]:
agent_executor.run("Who are the top 5 participants based on scores?")



> Entering new AgentExecutor chain...
Action: sql_db_list_tables
Action Input: 
Observation: clients, feedback_fact, learners_fact, order_duration_fact, orders_dimension, progress_fact, progress_report, skills_fact, table_meta_data
Thought: I should query the schema of the progress_fact table.
Action: sql_db_schema
Action Input: progress_fact
Observation: 
CREATE TABLE progress_fact (
	id VARCHAR, 
	order_id VARCHAR, 
	activity_id VARCHAR, 
	activity_name VARCHAR, 
	activity_status VARCHAR, 
	total_inputs_by_activity NUMERIC, 
	completed_inputs_by_activity NUMERIC, 
	progress_in_percent_by_activity DOUBLE PRECISION, 
	duration_min_by_activity DOUBLE PRECISION, 
	learner_status_by_order VARCHAR, 
	project_id VARCHAR, 
	project_name VARCHAR, 
	participants_email VARCHAR, 
	participants_name VARCHAR, 
	client_id VARCHAR, 
	activity_level VARCHAR, 
	created_at TIMESTAMP WITHOUT TIME ZONE, 
	is_current BOOLEAN DEFAULT true
)

/*
3 rows from progress_fact table:
id	order_id	activity_id	act

'The top 5 participants based on scores are Namratha Yeluri, Jyoti Raul, Suma Anusha Abbireddy, NIDHI HARISH, and Sita Maha LakshmiMukku.'

Here we have asked to fetch the score, thus it should fetch the scores instead of progress.

It should be mention in the context that when user asks the uestion regardin scores or skills then query
the skills_fact table

In [8]:
from dotenv import load_dotenv, find_dotenv
import os
from langchain.chat_models import ChatOpenAI
from langchain.agents import Tool, ZeroShotAgent, AgentExecutor
from langchain.tools import (
    Tool,
)  # Before import download it pip install duckduckgo-search
from langchain.callbacks import StreamlitCallbackHandler
import streamlit as st
from langchain.utilities import SQLDatabase, DuckDuckGoSearchAPIWrapper
from langchain_experimental.sql import SQLDatabaseChain
from langchain.chains import LLMMathChain, LLMChain
from langchain.memory import ConversationBufferMemory

from langchain.memory.chat_message_histories import SQLChatMessageHistory
from langchain.schema.output_parser import OutputParserException

from sqlalchemy.engine import URL

In [10]:
db = SQLDatabase.from_uri(connectionURL)
db_chain = SQLDatabaseChain.from_llm(
    ChatOpenAI(temperature=0, streaming=True, model="gpt-3.5-turbo-16k"),
    db,
    verbose=True,
)

tools = [
    # Tool.from_function(
    #     name="Search",
    #     func=search.run,
    #     description="useful for when you need to answer questions about current events",
    # ),
    # Tool.from_function(
    #     name="Calculator",
    #     func=llm_math_chain.run,
    #     description="useful for when you need to answer questions regarding calculations or maths",
    # ),
    Tool.from_function(
        name="Local DB",
        func=db_chain.run,
        description="Useful when we need to answer the questions regarding learners, clients, progress, skills, scores, etc.",
    ),
]

In [102]:
# The agent_scratchpad is where we add every thought or action the agent has already performed.
# All thoughts and actions (within the current agent executor chain) can then be accessed
# by the next thought-action-observation loop, enabling continuity in agent actions.
prefix = """You are a PotgrSQL expert. Given an input question, first create a syntactically correct PotgrSQL query to 
run, then look at the results of the query and return the answer to the input question. Unless the user specifies in 
the question a specific number of examples to obtain, query for at most 5 results using the LIMIT clause as per PotgrSQL. 
You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. 
Wrap each column name in double quotes ("") to denote them as delimited identifiers. Pay attention to use only the column names 
you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in 
which table. Pay attention to use date('now') function to get the current date, if the question involves "today".
Insert the is_current=True condition in all sql queries. If users asks the question regarding scores, skills, etc, then refer the 
skills_fact table. If user aks questions regarding activity_status, total_inputs, completed_inputs, progress_in_percent,
duration, learner's status then refer the progress_fact table


Example conversation:

User: Hey can you help me with something

Agent: Sure! What do you need help with?

User: I want you to perform queries on database when user ask questions. Also keep in mind that if a 
table contains is_current attribute then select only those records where is_current is True

Agent: Sure! Have noted your point.

User: When user ask the question related to the scores of learners, then refer the skills_fact table and find the 
total scores of a particular learners from the score attribute by performing group by on participant_email atribute.

Agent: Okay Got that.

User: When user ask the question related to the progress of learners, then refer the progress_in_percent_by_activity
and filter the records based on that learner giving his progress in all the project. And in case, if progress is asked for a
specific project then refer the project_name attribute of progress_fact table and give the progress of a specific project.

Run the following query:
SELECT participants_email, progress_in_percent
FROM progress_fact.

What is the progress of all the participants in the SQL-L2 Practice Set?

Run the following query:
SELECT participants_email, progress_in_percent
FROM progress_fact
WHERE project_name = 'SQL-L2 Practice Set' AND is_current = True;

Agent: Sure!

User: When user ask the question related to the learner's status, then refer the learner_status_by_order attribute of progress_fact
and filter the records based on that learner giving his status in all projects. And in case, if status is asked for a
specific project then refer the project_name attribute of progress_fact table and give the status of a specific project.

Agent: Sure!

User: Filter the top 5 participants based on their total scores in the Excel Assessment. In the given question, the excel assessment 
is the name of the project. So refer the project_name and filter the records based on that.

Agent: Sure!

User: When user asks to find the scores in percentage then refer the skills_fact table then find the 
total scores of a particular learners from the score attribute by performing group by on participant_email atribute
and find the out of scores from the total attribute by performing group by on participant_email atribute. And then calculate
the percentage based on total scores/out of scores.

Run the give query in the database:
SELECT participant_email, SUM(score) AS total_scores,
        sum(total) as out_of_scores,
        (sum(score)/sum(total))*100 as scores_in_percentage
        FROM skills_fact
        WHERE project_name = 'Excel Assessment' AND is_current = True
        GROUP BY participant_email
        ORDER BY total_scores DESC
        LIMIT 5;


Agent: Sure!


User: Filter the top 5 participants based on their scores in percentage in the Data Quality Assessment for the batch TRED-DE-07092023.
In the given question batch TRED-DE-07092023 is the substring of the description attribute in the orders_dimension table. Filter the 
records based on the batch name and then find the score

Run the give query in the database:
SELECT participant_email, SUM(score) AS total_scores,
        sum(total) as out_of_scores,
        (sum(score)/sum(total))*100 as scores_in_percentage
        FROM skills_fact sk
        join orders_dimension od 
        on sk.order_id =od.id
        WHERE sk.project_name = 'Excel Assessment' AND sk.is_current = true
        and od.description like '%TRED-DE-07092023%'
        GROUP BY participant_email
        ORDER BY total_scores DESC
        LIMIT 5;

Current conversation:

User: [input]

Agent:

Use the following format:
Question: Question here
SQLQuery: SQL Query to run
SQLResult: Result of the SQLQuery
Answer: Final answer here.
"""

suffix = """Begin!"

Question: {input}
{agent_scratchpad}"""

In [103]:
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["input", "agent_scratchpad"],
)

In [104]:
# print(prompt.template)

In [105]:
llm_chain = LLMChain(llm=ChatOpenAI(temperature=0, streaming=True, model="gpt-3.5-turbo-16k"), prompt=prompt)

In [106]:
agent = ZeroShotAgent(
    llm_chain=llm_chain,
    tools=tools,
    verbose=True,
)

In [107]:
agent_chain = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

In [20]:
agent_chain.run(input='How many tables are present in the database?')



> Entering new AgentExecutor chain...
Thought: To answer this question, I need to query the information_schema.tables table in the database.
Action: Local DB
Action Input: SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'public';

> Entering new SQLDatabaseChain chain...
SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'public';
SQLQuery:SELECT COUNT(*) FROM information_schema.tables WHERE table_schema = 'public';
SQLResult: [(9,)]
Answer:Final answer here: There are 9 tables in the public schema.
> Finished chain.

Observation: Final answer here: There are 9 tables in the public schema.
Thought:I now know the final answer.
Final Answer: There are 9 tables present in the database.

> Finished chain.


'There are 9 tables present in the database.'

In [21]:
agent_chain.run(input='How many total clients are there?')



> Entering new AgentExecutor chain...
Thought: To answer this question, I need to count the number of distinct clients in the database.
Action: Local DB
Action Input: SELECT COUNT(DISTINCT client_id) FROM clients

> Entering new SQLDatabaseChain chain...
SELECT COUNT(DISTINCT client_id) FROM clients
SQLQuery:SELECT COUNT(DISTINCT client_id) FROM clients
SQLResult: [(14,)]
Answer:Final answer here: There are 14 distinct client IDs in the clients table.
> Finished chain.

Observation: Final answer here: There are 14 distinct client IDs in the clients table.
Thought:I now know the final answer.
Final Answer: There are 14 total clients in the database.

> Finished chain.


'There are 14 total clients in the database.'

In [25]:
agent_chain.run("Filter the top 5 participants based on their total scores.")



> Entering new AgentExecutor chain...
Thought: To filter the top 5 participants based on their total scores, I need to query the skills_fact table and group the records by participant_email. Then, I can order the results by the sum of scores in descending order and limit the query to 5 results.

Action: Local DB
Action Input: SELECT participant_email, SUM(score) AS total_score
              FROM skills_fact
              WHERE is_current = True
              GROUP BY participant_email
              ORDER BY total_score DESC
              LIMIT 5;

> Entering new SQLDatabaseChain chain...
SELECT participant_email, SUM(score) AS total_score
              FROM skills_fact
              WHERE is_current = True
              GROUP BY participant_email
              ORDER BY total_score DESC
              LIMIT 5;
SQLQuery:SELECT participant_email, SUM(score) AS total_score
FROM skills_fact
WHERE is_current = True
GROUP BY participant_email
ORDER BY total_score DESC
LIMIT 5;
SQLResult: [('

'The top 5 participants based on their total scores are:\n1. soujanya.sastry@fractal.ai - 11669.0\n2. Pritam.Banik@fractal.ai - 10687.0\n3. kshitij.agarwal2@fractal.ai - 10595.0\n4. Akshay.Gajjar@fractal.ai - 10413.0\n5. venkat.yagati@fractal.ai - 9959.0'

In [39]:
agent_chain.run("Filter the top 5 participants based on their total scores in the Excel Assessment")



> Entering new AgentExecutor chain...
Thought: To answer this question, I need to query the skills_fact table and filter the records based on the project_name attribute. I will also need to group the records by participant_email and calculate the total scores for each participant.

Action: Local DB
Action Input: SELECT participant_email, SUM(score) AS total_scores
              FROM skills_fact
              WHERE project_name = 'Excel Assessment' AND is_current = True
              GROUP BY participant_email
              ORDER BY total_scores DESC
              LIMIT 5;

> Entering new SQLDatabaseChain chain...
SELECT participant_email, SUM(score) AS total_scores
              FROM skills_fact
              WHERE project_name = 'Excel Assessment' AND is_current = True
              GROUP BY participant_email
              ORDER BY total_scores DESC
              LIMIT 5;
SQLQuery:SELECT participant_email, SUM(score) AS total_scores
FROM skills_fact
WHERE project_name = 'Excel Asses

'The top 5 participants based on their total scores in the Excel Assessment are:\n1. mythily.r@tredence.com - 258.0\n2. ashim.gupta@tredence.com - 258.0\n3. sanjay.ls@tredence.com - 258.0\n4. prahart.boppana@tredence.com - 257.0\n5. kuldeep@tredence.com - 256.0'

In [67]:
agent_chain.run("Filter the top 5 participants based on their scores in percentage in the Excel Assessment")



> Entering new AgentExecutor chain...
Thought: We need to find the participants with the highest scores in percentage in the Excel Assessment.
Action: Local DB
Action Input: 
```sql
SELECT participant_email, SUM(score) AS total_scores,
        sum(total) as out_of_scores,
        (sum(score)/sum(total))*100 as scores_in_percentage
        FROM skills_fact
        WHERE project_name = 'Excel Assessment' AND is_current = True
        GROUP BY participant_email
        ORDER BY scores_in_percentage DESC
        LIMIT 5;
```

> Entering new SQLDatabaseChain chain...
```sql
SELECT participant_email, SUM(score) AS total_scores,
        sum(total) as out_of_scores,
        (sum(score)/sum(total))*100 as scores_in_percentage
        FROM skills_fact
        WHERE project_name = 'Excel Assessment' AND is_current = True
        GROUP BY participant_email
        ORDER BY scores_in_percentage DESC
        LIMIT 5;
```
SQLQuery:SELECT participant_email, SUM(score) AS total_scores,
        sum(to

'The top 5 participants based on their scores in percentage in the Excel Assessment are:\n\n1. gorrepotu.sai@tredence.com - 100.0%\n2. sandeep.kumar@tredence.com - 100.0%\n3. himanshu.mittal@tredence.com - 95.71%\n4. ashim.gupta@tredence.com - 95.56%\n5. mythily.r@tredence.com - 95.56%'

In [68]:
# Now change the project_name and check
agent_chain.run("Filter the top 5 participants based on their scores in percentage in the Data Quality")



> Entering new AgentExecutor chain...
Thought: To answer this question, I need to query the skills_fact table and filter the records based on the project_name attribute. Then, I need to calculate the scores in percentage for each participant and order the results by the scores in percentage in descending order. Finally, I can limit the results to the top 5 participants.

Action: Local DB
Action Input: 
```sql
SELECT participant_email, 
       (SUM(score) / SUM(total)) * 100 AS scores_in_percentage
FROM skills_fact
WHERE project_name = 'Data Quality' AND is_current = True
GROUP BY participant_email
ORDER BY scores_in_percentage DESC
LIMIT 5;
```


> Entering new SQLDatabaseChain chain...
```sql
SELECT participant_email, 
       (SUM(score) / SUM(total)) * 100 AS scores_in_percentage
FROM skills_fact
WHERE project_name = 'Data Quality' AND is_current = True
GROUP BY participant_email
ORDER BY scores_in_percentage DESC
LIMIT 5;
```

SQLQuery:

OperationalError: (psycopg2.OperationalError) server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.
server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.

[SQL: SELECT clients.client_id, clients.client_name, clients.client_email, clients.created_at, clients.is_current 
FROM clients 
 LIMIT %(param_1)s]
[parameters: {'param_1': 3}]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [109]:
# Now change the project_name and check
agent_chain.run(
    "Filter the top 5 participants based on their scores in percentage in the Data Quality Assessment for the batch TRED-DE-07092023"
)



> Entering new AgentExecutor chain...
Thought: To answer this question, I need to find the scores in percentage for the Data Quality Assessment for the batch TRED-DE-07092023. I will need to join the skills_fact table with the orders_dimension table to filter the records based on the batch name.

Action: Local DB
Action Input: 
SELECT participant_email, SUM(score) AS total_scores,
        sum(total) as out_of_scores,
        (sum(score)/sum(total))*100 as scores_in_percentage
        FROM skills_fact sk
        join orders_dimension od 
        on sk.order_id =od.id
        WHERE sk.project_name = 'Data Quality Assessment' AND sk.is_current = true
        and od.description like '%TRED-DE-07092023%'
        GROUP BY participant_email
        ORDER BY scores_in_percentage DESC
        LIMIT 5;


> Entering new SQLDatabaseChain chain...
SELECT participant_email, SUM(score) AS total_scores,
        sum(total) as out_of_scores,
        (sum(score)/sum(total))*100 as scores_in_percentage


'The top 5 participants based on their scores in percentage in the Data Quality Assessment for the batch TRED-DE-07092023 are:\n\n1. Participant Email: sandeep.kumar@tredence.com\n   Scores in Percentage: 100.0%\n\n2. Participant Email: srijan.patel@tredence.com\n   Scores in Percentage: 100.0%\n\n3. Participant Email: abhishek.s01@tredence.com\n   Scores in Percentage: 100.0%\n\n4. Participant Email: hazik.magrey@tredence.com\n   Scores in Percentage: 100.0%\n\n5. Participant Email: harneet.kohli@tredence.com\n   Scores in Percentage: 96.67%'

We can observe that when Data Quality is mentioned, then in that case it gave the error, but in the case of Data Quality Assessment it gave the output. Mention in the prompt that use like with project_name.

In [101]:
# Now change the project_name and check
agent_chain.run("Filter the top 5 participants based on their scores in percentage in the Data Quality Assessment")



> Entering new AgentExecutor chain...
Thought: To answer this question, I need to query the skills_fact table and filter the records based on the project_name attribute. Then, I need to calculate the scores in percentage by dividing the total scores by the out of scores and multiplying by 100. Finally, I need to order the results by scores in percentage and limit the results to 5.

Action: Local DB
Action Input: 
SELECT participant_email, 
       SUM(score) AS total_scores,
       SUM(total) AS out_of_scores,
       (SUM(score)/SUM(total))*100 AS scores_in_percentage
FROM skills_fact
WHERE project_name = 'Data Quality Assessment' AND is_current = True
GROUP BY participant_email
ORDER BY scores_in_percentage DESC
LIMIT 5;


> Entering new SQLDatabaseChain chain...
SELECT participant_email, 
       SUM(score) AS total_scores,
       SUM(total) AS out_of_scores,
       (SUM(score)/SUM(total))*100 AS scores_in_percentage
FROM skills_fact
WHERE project_name = 'Data Quality Assessment' AND

'The top 5 participants based on their scores in percentage in the "Data Quality Assessment" project are:\n1. abhishek.s01@tredence.com - Scores in percentage: 100%\n2. srijan.patel@tredence.com - Scores in percentage: 100%\n3. sandeep.kumar@tredence.com - Scores in percentage: 100%\n4. hazik.magrey@tredence.com - Scores in percentage: 100%\n5. harneet.kohli@tredence.com - Scores in percentage: 96.67%'